# Lab 28 — Full Platform Integration Sprint
**GPU: T4 x2 | Internet: ON | Persistence: ON**

Chạy từng cell theo thứ tự từ trên xuống dưới.

## Cell 1 — Install Dependencies
> `pyngrok` cài bằng pip — không cần download binary.

In [ ]:
!pip install -q vllm fastapi uvicorn mlflow sentence-transformers requests pyngrok

import subprocess, os

# Kiểm tra GPU
import torch
print(f'✅ GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT FOUND"}')
print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'   Compute: {torch.cuda.get_device_capability(0)}')

# Kiểm tra pyngrok
from pyngrok import ngrok
print(f'✅ pyngrok installed OK')


## Cell 2 — Authenticate Ngrok
> Token lấy miễn phí tại https://dashboard.ngrok.com/get-started/your-authtoken
> Thêm vào Kaggle Secrets (Settings → Add-ons → Secrets) với key `NGROK_AUTHTOKEN`

In [ ]:
# ──────────────────────────────────────────────────────────────────
# Cách lấy NGROK_AUTHTOKEN (miễn phí):
# 1. Đăng ký tại https://dashboard.ngrok.com/signup
# 2. Vào https://dashboard.ngrok.com/get-started/your-authtoken
# 3. Copy token → thêm vào Kaggle Secrets:
#    Kaggle Notebook → Settings → Add-ons → Secrets → Add (tên: NGROK_AUTHTOKEN)
# ──────────────────────────────────────────────────────────────────
from kaggle_secrets import UserSecretsClient
from pyngrok import ngrok, conf

try:
    secrets = UserSecretsClient()
    NGROK_TOKEN = secrets.get_secret("NGROK_AUTHTOKEN")
    print(f'✅ Ngrok token loaded from Kaggle Secrets ({len(NGROK_TOKEN)} chars)')
except Exception:
    # Fallback: paste token thủ công vào đây nếu không dùng Kaggle Secrets
    NGROK_TOKEN = ""  # ← PASTE TOKEN CỦA BẠN VÀO ĐÂY NẾU CẦN
    if not NGROK_TOKEN:
        print("❌ Không tìm thấy NGROK_AUTHTOKEN!")
        print("   → Vào https://dashboard.ngrok.com/get-started/your-authtoken")
        print("   → Thêm vào Kaggle Secrets với tên: NGROK_AUTHTOKEN")
        raise SystemExit("Cần có Ngrok token để tiếp tục.")

ngrok.set_auth_token(NGROK_TOKEN)
print('✅ Ngrok authenticated')


## Cell 3 — Start vLLM Server
> `--enforce-eager` bắt buộc cho T4 (compute 7.5) để tránh flashinfer JIT crash.

In [ ]:
import subprocess, threading, time, requests, os

# Kill tiến trình vLLM cũ nếu còn
subprocess.run(['pkill', '-f', 'vllm.entrypoints'], capture_output=True)
time.sleep(2)

def run_vllm():
    subprocess.run([
        'python', '-m', 'vllm.entrypoints.openai.api_server',
        '--model', 'Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4',
        '--port', '8001',
        '--max-model-len', '2048',
        '--gpu-memory-utilization', '0.90',
        '--host', '0.0.0.0',
        '--enforce-eager',  # Bắt buộc cho T4 compute 7.5 — tránh flashinfer JIT crash
    ])

print('Starting vLLM (--enforce-eager, ~3-4 phút)...')
thread = threading.Thread(target=run_vllm, daemon=True)
thread.start()

for i in range(24):  # tối đa 4 phút
    time.sleep(10)
    try:
        resp = requests.get('http://localhost:8001/v1/models', timeout=3)
        if resp.status_code == 200:
            models = resp.json().get('data', [])
            print(f'✅ vLLM ready after {(i+1)*10}s')
            print(f'   Models: {[m["id"] for m in models]}')
            break
    except:
        print(f'  Loading... {(i+1)*10}s')
else:
    print('⚠️ vLLM timeout — check logs above')


## Cell 4 — Tạo Ngrok Tunnel cho vLLM
> Copy `VLLM_NGROK_URL=...` vào `.env` local.

In [ ]:
from pyngrok import ngrok
import time

# Kill tunnel cũ nếu còn
ngrok.kill()
time.sleep(1)

# Tạo tunnel cho vLLM
vllm_tunnel = ngrok.connect(8001, "http")
VLLM_URL = vllm_tunnel.public_url
# Đảm bảo dùng https
if VLLM_URL.startswith('http://'):
    VLLM_URL = VLLM_URL.replace('http://', 'https://', 1)

print(f'✅ vLLM URL: {VLLM_URL}')
print(f'\n👉 Paste vào .env local:')
print(f'   VLLM_NGROK_URL={VLLM_URL}')


## Cell 5 — Start Embedding Service
> Model: `BAAI/bge-small-en-v1.5` — 384 dims

In [ ]:
from fastapi import FastAPI
from sentence_transformers import SentenceTransformer
import uvicorn, threading

embed_app = FastAPI(title='Embedding Service')
print('Loading BAAI/bge-small-en-v1.5...')
embed_model = SentenceTransformer('BAAI/bge-small-en-v1.5')
print('✅ Embedding model loaded')

@embed_app.post('/embed')
def embed(data: dict):
    texts = data.get('texts', [])
    if not texts:
        return {'embeddings': [], 'error': 'No texts provided'}
    return {'embeddings': embed_model.encode(texts, normalize_embeddings=True).tolist(),
            'count': len(texts)}

@embed_app.get('/health')
def health():
    return {'status': 'ok', 'model': 'BAAI/bge-small-en-v1.5'}

threading.Thread(
    target=lambda: uvicorn.run(embed_app, host='0.0.0.0', port=8002, log_level='warning'),
    daemon=True
).start()
print('✅ Embedding server started on port 8002')


## Cell 6 — Tạo Ngrok Tunnel cho Embedding
> Copy `EMBED_NGROK_URL=...` vào `.env` local.

In [ ]:
import requests, time

time.sleep(2)  # Chờ uvicorn khởi động

# Tạo tunnel cho Embedding
embed_tunnel = ngrok.connect(8002, "http")
EMBED_URL = embed_tunnel.public_url
if EMBED_URL.startswith('http://'):
    EMBED_URL = EMBED_URL.replace('http://', 'https://', 1)

print(f'✅ Embedding URL: {EMBED_URL}')
print(f'\n👉 Paste vào .env local:')
print(f'   EMBED_NGROK_URL={EMBED_URL}')

# Test embedding service
try:
    resp = requests.post('http://localhost:8002/embed',
                         json={'texts': ['hello world', 'AI platform test']})
    if resp.status_code == 200:
        data = resp.json()
        print(f"\n✅ Embedding test OK: count={data['count']}, dim={len(data['embeddings'][0])}")
    else:
        print('⚠️ Embedding test failed:', resp.text)
except Exception as e:
    print(f'❌ Error: {e}')


## Cell 7 — MLflow Tracking

In [ ]:
import mlflow

mlflow.set_tracking_uri('./mlruns')
mlflow.set_experiment('lab28-integration')

with mlflow.start_run(run_name='vllm-serving-v1') as run:
    mlflow.log_param('model', 'Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4')
    mlflow.log_param('max_model_len', 2048)
    mlflow.log_param('gpu_memory_utilization', 0.90)
    mlflow.log_param('embedding_model', 'BAAI/bge-small-en-v1.5')
    mlflow.log_param('enforce_eager', True)
    mlflow.log_metric('embedding_dim', 384)
    mlflow.set_tag('vllm_url', VLLM_URL)
    mlflow.set_tag('embed_url', EMBED_URL)
    mlflow.set_tag('status', 'production')
    mlflow.set_tag('lab', 'lab28')
    run_id = run.info.run_id

print(f'✅ MLflow run_id={run_id}')
print(f'   Experiment: lab28-integration')


## Cell 8 — Test Full Pipeline

In [ ]:
import requests

print('=' * 55)
print('  TESTING FULL PIPELINE via Ngrok URLs')
print('=' * 55)

# Test vLLM
print('\n[1] Testing vLLM inference...')
try:
    resp = requests.post(f'{VLLM_URL}/v1/chat/completions', json={
        'model': 'Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4',
        'messages': [{'role': 'user', 'content': 'Reply with exactly: Lab 28 OK'}],
        'max_tokens': 10
    }, timeout=60)
    if resp.status_code == 200:
        answer = resp.json()['choices'][0]['message']['content']
        latency = resp.elapsed.total_seconds() * 1000
        print(f"   ✅ vLLM OK | Answer: '{answer}' | Latency: {latency:.0f}ms")
    else:
        print(f'   ⚠️ vLLM HTTP {resp.status_code}: {resp.text[:200]}')
except Exception as e:
    print(f'   ❌ vLLM Error: {e}')

# Test Embedding
print('\n[2] Testing Embedding service...')
try:
    resp = requests.post(f'{EMBED_URL}/embed', json={
        'texts': ['platform engineering', 'AI infrastructure']
    }, timeout=30)
    if resp.status_code == 200:
        data = resp.json()
        print(f"   ✅ Embed OK | count={data['count']}, dim={len(data['embeddings'][0])}")
    else:
        print(f'   ⚠️ Embed HTTP {resp.status_code}')
except Exception as e:
    print(f'   ❌ Embed Error: {e}')

print('\n' + '=' * 55)
print('\n📋 Copy to local .env:')
print(f'   VLLM_NGROK_URL={VLLM_URL}')
print(f'   EMBED_NGROK_URL={EMBED_URL}')


## Cell 9 — Keep Alive
> Giữ session Kaggle không timeout. Bấm **Interrupt** để dừng.

In [ ]:
import time, requests

print('🟢 Notebook running. Active Ngrok URLs:')
print(f'   vLLM:      {VLLM_URL}')
print(f'   Embedding: {EMBED_URL}')
print('\nKeeping session alive (Interrupt Kernel to stop)...')

counter = 0
while True:
    time.sleep(300)
    counter += 1
    try:
        requests.get('http://localhost:8001/v1/models', timeout=2)
    except:
        pass
    print(f'  [keepalive] {counter * 5} min | vLLM: {VLLM_URL}')
